In [ ]:
sys.path.insert(0, snakemake.input.data2dd)

In [ ]:
import pandas as pd
from data2dd_funcs import wrapdd

In [ ]:
year = snakemake.wildcards.year
date_range = snakemake.params.date_range
date_range = [year + "-" + date for date in date_range]
date_range = pd.date_range(date_range[0], date_range[1] + ' 23', freq='h')

In [ ]:
aggregated_regions = snakemake.params.aggregated_regions
aggregated_regions

In [ ]:
# ev_scenarios.csv # contain the number of EV in each zone
ev_scenarios = snakemake.input["ev_scenarios"]
evs = pd.read_csv(ev_scenarios, index_col=0).loc[snakemake.params.ev_scenario][aggregated_regions].T.reset_index()
wrapdd(evs, "par_vehicles", "parameter", outfile=snakemake.output.vehicles)

In [ ]:
# ev_scalars.tsv
import shutil
shutil.copy2(snakemake.input[3], snakemake.output[3])

In [ ]:
# EV_grid_connected_power_kW.csv

s_ev_pcap = pd.read_csv(snakemake.input[3], sep=' ', header=None, index_col=0).iloc[:,0].str.strip('/').astype(float).loc['s_ev_pcap']

connected_vehicles = pd.read_csv(snakemake.input[0]) #, sep='\t', header=None, index_col=0)
connected_vehicles = connected_vehicles.charging_cap.iloc[:len(date_range)].round(3).div(1000).div(s_ev_pcap).reset_index().astype({'index':str}) #.to_csv(snakemake.output[0], sep = '\t', header = False)
wrapdd(connected_vehicles, "par_grid_connected", "parameter", outfile=snakemake.output[0])

In [ ]:
# demand_driving_MWh.tsv
demand_driving = pd.read_csv(snakemake.input[1]) #, sep='\t', header=None, index_col=0)
demand_driving = demand_driving.set_index(demand_driving.reset_index()["index"].astype('str') + '.EV')['battery discharge kWh'].iloc[:len(date_range)].div(1000).reset_index() #.to_csv(snakemake.output[1], sep = '\t', header = False)
wrapdd(demand_driving, "par_driving_demand", "parameter", outfile=snakemake.output[1])

In [ ]:
# demand_ev_charging_MWh.tsv
demand_ev_charging = pd.read_csv(snakemake.input[2]) #, sep='\t', header=None, index_col=0)
demand_ev_charging = demand_ev_charging.charge_battery.iloc[:len(date_range)].round(6).div(1000).reset_index().astype({'index':str}) #.to_csv(snakemake.output[2], sep = '\t', header = False)
wrapdd(demand_ev_charging, "par_ev_charging", "parameter", outfile=snakemake.output[2])